# Lab 6.2 &mdash; Retrieval as a Tool the Agent Chooses

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Decide <em>whether</em> to retrieve &mdash; and find the questions where retrieving hurts
- Price always-retrieve: the tokens, and the noise it puts next to the real context
- Write the query in the corpus's vocabulary instead of passing the user's words through
- Compare the pipeline and the agent on the same questions

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds directly on Lab 6.1's retriever.** Same index, same scoring. What changes is
> who decides when it runs and what it is asked.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-6-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Two short operating documents. Read 3.2: the rule and the exception that qualifies it are
# adjacent sentences, which is the whole of Lab 6.1's first lesson. Note also what is NOT
# here -- there is nothing about FX or hedging anywhere, and Lab 6.4 needs that gap.

DOCS = {
    "ops-runbook-v4.md": """
## 3.1 Insufficient funds
A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours. If the retry also fails,
notify the client desk. Operations must not fund the account manually.

## 3.2 Limit breaches
Payments above USD 500,000 require Treasury approval before release. This does not apply to
intra-group transfers, which settle same-day without any approval.

## 3.3 Invalid beneficiary details
A payment returned INVALID_IBAN is returned to the originator with code R04. Beneficiary
details are never repaired in-house.

## 3.4 Sanctions review
A payment held for SANCTIONS_REVIEW is decided by Compliance. Operations must not release or
cancel it under any circumstances.
""",
    "escalation-policy-v2.md": """
## 1 Approval authority
A duty manager may approve a release up to USD 250,000. Above that figure Treasury approval is
required, and must be recorded against the payment reference.

## 2 Escalation timers
If an approver has not responded within 15 minutes, escalate to the Treasury lead, and after a
further 15 minutes to the head of operations.
""",
}

print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS.values())} characters")

In [ ]:
# ------------------------------------------------- carried forward from Lab 6.1 (nothing to fill in)
import re

SECTION_RE = re.compile(r"^##\s+(.*)$", re.M)
STOP = set("""a an the of for is are was were do does did what which who this that these those it
its to in on at by with from about and or not no be been have has had can could should would will
you your we our i me my how why when where there here as if then than so such only just also very
more most some any other""".split())

def terms(text):
    return {w for w in re.findall(r"[a-z0-9_]+", (text or "").lower())
            if w not in STOP and len(w) > 1}

def chunk_by_section(source, text):
    out, parts = [], SECTION_RE.split(text)
    for i in range(1, len(parts) - 1, 2):
        heading, body = parts[i].strip(), " ".join(parts[i + 1].split())
        out.append({"source": source, "section": heading, "text": heading + " -- " + body})
    return out

INDEX = [c for source, text in DOCS.items() for c in chunk_by_section(source, text)]

def similarity(query, chunk):
    q = terms(query)
    return len(q & terms(chunk["text"])) / len(q) if q else 0.0

def search(query, index=None, k=4, floor=0.0):
    index = INDEX if index is None else index
    scored = sorted(((similarity(query, c), c) for c in index), key=lambda sc: -sc[0])
    return [{"score": round(s, 3), **c} for s, c in scored[:k] if s >= floor]

print(f"index: {len(INDEX)} chunks")

## Concept

A pipeline retrieves once, always, with the user's exact words. That is one decision, made at
build time, applied to every question.

An agent makes three decisions per question, and this lab builds the first two:

- **whether** &mdash; some questions are answered by the conversation, or by arithmetic
- **what to ask for** &mdash; users write in their words, corpora in the organisation's

The third, *whether to ask again*, is Lab 6.3.

## Section 1 &mdash; Whether to retrieve at all

Three kinds of question, and only one of them wants the corpus.

In [ ]:
QUESTIONS = [
    # (question, does it need the corpus?)
    ("What approval does a payment above USD 500,000 need?",   True),
    ("What happens when a payment comes back INVALID_IBAN?",   True),
    ("Who decides on a payment held for sanctions review?",    True),
    ("How long before an unanswered approval escalates?",      True),
    ("What is 990,000 minus 500,000?",                         False),
    ("Calculate the difference between the amount and the limit.", False),
    ("Summarise what we just agreed.",                         False),
    ("What did I ask you a moment ago?",                       False),
]

CONVERSATION_HINTS = ("we just", "you said", "a moment ago", "earlier you", "we agreed",
                      "summarise what we", "recap")
ARITHMETIC_HINTS   = ("plus", "minus", "times", "calculate", "subtract", "difference between",
                      "how much is")

def needs_corpus(question: str) -> bool:
    """Should the agent go and read something for this question?

    Two kinds of question do not need the corpus: the ones the conversation has already
    answered, and the ones that are pure arithmetic.
    """
    low = (question or "").lower()
    # TODO: False for those two kinds, True for everything else.
    return BLANK

In [ ]:
# --- Self-check: Section 1
check("it gets every question in the set right",
      lambda: all(needs_corpus(q) is expected for q, expected in QUESTIONS))
check("a policy question retrieves",
      lambda: needs_corpus("Who decides on a payment held for sanctions review?") is True)
check("arithmetic does not",
      lambda: needs_corpus("What is 990,000 minus 500,000?") is False)
check("nor does a question about the conversation",
      lambda: needs_corpus("Summarise what we just agreed.") is False)
check("half the set needs no corpus at all",
      lambda: sum(1 for _, e in QUESTIONS if not e) == 4,
      "that fraction is the whole argument -- a pipeline retrieves for all eight")
check("an empty question does not crash the decision",
      lambda: isinstance(needs_corpus(""), bool))

## Section 2 &mdash; What always-retrieve costs

Two things, and the second is the one that does not show up on an invoice: tokens spent, and
irrelevant policy prose sitting next to the real context while the model tries to answer.

In [ ]:
def tokens_of(results) -> int:
    """A rough token count for retrieved text."""
    return sum(len(r["text"]) // 4 for r in results)


def run_pipeline(question: str) -> dict:
    """Always retrieve, with the user's words, exactly once."""
    hits = search(question, k=4)
    return {"retrieved": hits, "tokens": tokens_of(hits), "asked_for_it": True}


def run_agentic(question: str) -> dict:
    """Retrieve only when the question needs the corpus."""
    if not needs_corpus(question):
        return {"retrieved": [], "tokens": 0, "asked_for_it": False}
    hits = search(question, k=4)
    return {"retrieved": hits, "tokens": tokens_of(hits), "asked_for_it": True}


def wasted_tokens(runner) -> int:
    """Tokens spent retrieving for questions that did not need the corpus."""
    total = 0
    for question, expected in QUESTIONS:
        if not expected:
            # TODO: what this runner spent on a question that needed nothing
            total += BLANK
    return total

In [ ]:
# --- Self-check: Section 2
check("the pipeline retrieves for every question",
      lambda: all(run_pipeline(q)["asked_for_it"] for q, _ in QUESTIONS))
check("the agent retrieves for exactly the four that need it",
      lambda: sum(1 for q, _ in QUESTIONS if run_agentic(q)["asked_for_it"]) == 4)
check("the pipeline wastes real tokens on the other four",
      lambda: wasted_tokens(run_pipeline) > 0)
check("the agent wastes none",
      lambda: wasted_tokens(run_agentic) == 0)
check("both answer the corpus questions identically",
      lambda: all(run_pipeline(q)["retrieved"] == run_agentic(q)["retrieved"]
                  for q, e in QUESTIONS if e),
      "the agent is not retrieving less well -- it is retrieving less often")
check("and the noise is the part that does not show on the invoice",
      lambda: len(run_pipeline("Summarise what we just agreed.")["retrieved"]) == 4,
      "four chunks of policy prose, competing with the conversation the answer is actually in")

def _cost():
    p, a = sum(run_pipeline(q)["tokens"] for q, _ in QUESTIONS), \
           sum(run_agentic(q)["tokens"] for q, _ in QUESTIONS)
    print(f"  pipeline  {p:>5} retrieved tokens   ({wasted_tokens(run_pipeline)} of them wasted)")
    print(f"  agent     {a:>5} retrieved tokens   ({wasted_tokens(run_agentic)} of them wasted)")
guard(_cost)

## Section 3 &mdash; Ask in the corpus's words

Users describe their situation. Documents describe the organisation's rules. The agent's job is
translation &mdash; and it is the cheapest retrieval improvement there is, because it changes nothing
about the index.

In [ ]:
# What people say -> what the documents call it
VOCAB = {
    "bounce":          "INSUFFICIENT_FUNDS retry",
    "bounced":         "INSUFFICIENT_FUNDS retry",
    "push it through": "release Treasury approval",
    "push through":    "release Treasury approval",
    "wrong account":   "INVALID_IBAN beneficiary originator",
    "bad iban":        "INVALID_IBAN beneficiary originator",
    "on hold":         "SANCTIONS_REVIEW Compliance",
    "stuck":           "SANCTIONS_REVIEW Compliance",
    "chase":           "escalate approver Treasury lead",
    "our own entities": "intra-group transfers",
}

def rewrite(question: str) -> str:
    """The query the agent sends, which is the question plus the vocabulary it implies."""
    low = (question or "").lower()
    extra = [v for k, v in VOCAB.items() if k in low]
    # TODO: keep the user's words AND add the corpus vocabulary they imply.
    # Dropping the original loses everything the table does not cover.
    return BLANK

In [ ]:
# --- Self-check: Section 3
VAGUE = [
    ("Why did this one bounce, and do we try again?",            "3.1"),
    ("The client gave us the wrong account number. Now what?",   "3.3"),
    ("It is stuck. Who decides?",                                "3.4"),
    ("Can we push a big one through between our own entities?",  "3.2"),
]

def best_section(query):
    hits = search(query, k=1)
    return hits[0]["section"] if hits else None

check("the rewrite keeps the user's own words",
      lambda: rewrite("Why did this one bounce?").startswith("Why did this one bounce?"),
      "the table cannot cover everything; dropping the original loses whatever it missed")
check("and adds the corpus vocabulary",
      lambda: "INSUFFICIENT_FUNDS" in rewrite("Why did this one bounce?"))
check("a question with no match is passed through unchanged",
      lambda: rewrite("Who approves a release?") == "Who approves a release?")
check("raw vague questions mostly miss",
      lambda: sum(1 for q, want in VAGUE if (best_section(q) or "").startswith(want)) <= 2)
check("rewritten, they all land on the right section",
      lambda: all((best_section(rewrite(q)) or "").startswith(want) for q, want in VAGUE),
      "same index, same scoring, same k -- the only change is who wrote the query")

def _rewrites():
    for q, want in VAGUE:
        print(f"  {q}")
        print(f"      raw       -> {best_section(q)}")
        print(f"      rewritten -> {best_section(rewrite(q))}   (want {want})")
guard(_rewrites)

## Run it for real

Let the model do the rewriting instead of a lookup table &mdash; which is what you would actually
ship, because no table survives contact with real users.

In [ ]:
if llm_ready():
    def _model_rewrite():
        vocab_hint = ("The documents use terms like: INSUFFICIENT_FUNDS, INVALID_IBAN, "
                      "SANCTIONS_REVIEW, Treasury approval, intra-group transfer, escalation.")
        for q, want in VAGUE:
            query = ask(f"{vocab_hint}\n\nRewrite this into a search query using those terms. "
                        f"Reply with the query alone.\n\n{q}",
                        system="Reply with a search query and nothing else.")
            got = best_section((query or "").strip())
            flag = "ok " if (got or "").startswith(want) else "MISS"
            print(f"  [{flag}] {q[:44]:46} -> {got}")
    guard(_model_rewrite)

### Read it

If the model's rewrites land as well as the lookup table's, you have something that generalises to
questions you never enumerated &mdash; at the cost of one model call before every retrieval. That is a
real trade, and Lab 6.5 is where you price it.

Watch for the failure mode too: a rewrite that invents a term the corpus does not contain retrieves
*worse* than the raw question. Same lesson as Module 4 &mdash; a description, or a query, can attract
the wrong thing as easily as the right one.

In [ ]:
score()

## Your turn

1. `needs_corpus` is a keyword list, so it fails on any phrasing you did not think of. Write three
   questions that should not retrieve and that it gets wrong. What does that tell you about
   shipping this as a rule rather than as a model call?
2. There is a third answer besides yes and no: *retrieve, but only if the first attempt at
   answering is thin*. Sketch it, and say what it costs in latency.
3. Combine the two halves: rewrite first, then decide whether to retrieve based on how well the
   rewritten query scores. Does that ordering help, or have you just moved the guess?